# Train the pneumonia screening CNN (DenseNet-169 transfer learning)

Run this **once** on Google Colab with a free GPU, then download
`pneumonia_densenet169.weights.h5` and drop it into `mis/ai-service/model/`.

**Colab menu → Runtime → Change runtime type → Hardware accelerator: GPU.**

Method mirrors Varshni et al., *"Pneumonia Detection Using CNN based Feature
Extraction"* (IEEE, 2019): a DenseNet-169 backbone pre-trained on ImageNet is
used as a feature extractor. Here we attach a small trainable classifier head
and fine-tune, rather than a separate SVM, because it is simpler to serve and
gives comparable accuracy on the binary task.

Dataset: **Chest X-Ray Images (Pneumonia)** — Kermany et al., ~5,863 images.

In [ ]:
!pip -q install kagglehub
import kagglehub, os
path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
DATA = os.path.join(path, 'chest_xray')
print('dataset at:', DATA)
print(os.listdir(DATA))

In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import DenseNet169
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras import layers, models

IMG = 224
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
def ds(split, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(DATA, split),
        labels='inferred', label_mode='binary',
        class_names=['NORMAL', 'PNEUMONIA'],
        image_size=(IMG, IMG), batch_size=BATCH, shuffle=shuffle, seed=42)

train_ds = ds('train', True)
val_ds   = ds('val', False)
test_ds  = ds('test', False)

prep = lambda x, y: (preprocess_input(x), y)
aug = tf.keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.05),
                           layers.RandomZoom(0.1), layers.RandomContrast(0.1)])

train = train_ds.map(lambda x, y: (aug(x), y)).map(prep).prefetch(AUTOTUNE)
val   = val_ds.map(prep).prefetch(AUTOTUNE)
test  = test_ds.map(prep).prefetch(AUTOTUNE)

In [ ]:
# class weights - the training set is ~3:1 pneumonia:normal
import numpy as np, collections
counts = collections.Counter()
for _, y in train_ds.unbatch():
    counts[int(y.numpy()[0])] += 1
total = sum(counts.values())
class_weight = {c: total / (2 * n) for c, n in counts.items()}
print('counts', dict(counts), '| class_weight', class_weight)

In [ ]:
base = DenseNet169(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3), pooling='avg')
base.trainable = False  # stage 1: feature extractor frozen

model = models.Sequential([
    base,
    layers.Dropout(0.3),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid'),
])
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.summary()

In [ ]:
cb = [tf.keras.callbacks.EarlyStopping(monitor='val_auc', mode='max', patience=3, restore_best_weights=True)]
model.fit(train, validation_data=val, epochs=8, class_weight=class_weight, callbacks=cb)

In [ ]:
# stage 2: unfreeze the last dense block and fine-tune at a low LR
base.trainable = True
for l in base.layers[:-40]:
    l.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='binary_crossentropy',
              metrics=['accuracy', tf.keras.metrics.AUC(name='auc')])
model.fit(train, validation_data=val, epochs=5, class_weight=class_weight, callbacks=cb)

In [ ]:
# evaluate on the held-out test set
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
y_true = np.concatenate([y.numpy().ravel() for _, y in test])
y_prob = model.predict(test).ravel()
y_pred = (y_prob >= 0.5).astype(int)
print('AUC :', round(roc_auc_score(y_true, y_prob), 4))
print('Confusion matrix [NORMAL, PNEUMONIA]:')
print(confusion_matrix(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=['NORMAL', 'PNEUMONIA']))

In [ ]:
import json
# Save WEIGHTS ONLY (not the full model) - this is immune to Keras version
# mismatches between Colab and wherever ai-service runs. predictor.py rebuilds
# the identical architecture and loads these weights.
model.save_weights('pneumonia_densenet169.weights.h5')
json.dump({'model_name': 'DenseNet169-TL', 'model_version': '1.0',
           'auc': float(roc_auc_score(y_true, y_prob)),
           'trained_on': 'chest-xray-pneumonia (Kermany)'},
          open('model_meta.json', 'w'), indent=2)

from google.colab import files
files.download('pneumonia_densenet169.weights.h5')
files.download('model_meta.json')
print('Done. Put both files in  mis/ai-service/model/  then restart the AI service.')
